# 04.05 · Excel de la Preparatoria CCI

Se genera `data/processed/4. Preparatoria CCI.xlsx`: una fila por documento de la agenda, con
las columnas A–R de la hoja `CCI-2026` del Excel institucional *Preparación CCI y CFE - VPE y
GPID*, más `Revisar` y `Archivo fuente`. Las filas se copian después al Excel institucional; este
archivo no lo modifica.

| Columnas | Origen |
|---|---|
| A, B. Fecha y No. | Carpeta de la sesión y número en la agenda (`04.01`) |
| C, E–N. Operación, tipo, nombre, país, riesgo, producto, monto, FV, CT, agenda | Ficha (`04.02`) |
| D. Retroalimentación DADMI | Vacía: se completa a mano (en `Revisar` se sugiere 1 si hay insumo Evid+Recom) |
| O, P. Aprendizajes y Recomendaciones | IDO y aprobación: Input CCI o preparatoria anterior (`04.03`); líneas y renovaciones: aporte al desarrollo (`04.02`) |
| Q, R. Alertas y preguntas | `04.04` |

El semáforo de las recomendaciones (🟢 🟡 🔴 ⚪) es un proceso aparte: las recomendaciones se
copian sin semáforo. El notebook no usa la API y el Excel se reemplaza en cada ejecución.

## 1. LOAD LIBRARIES

pandas para la tabla y openpyxl para el Excel, como en el `02.05`. Las rutas se toman de
`caf_input_cci_preparatoria/config.py`.

In [ ]:
from datetime import date
import json
from pathlib import Path
import sys

from IPython.display import display
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.workbook.defined_name import DefinedName
from openpyxl.worksheet.datavalidation import DataValidation
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # para importar el paquete caf_input_cci_preparatoria

from caf_input_cci_preparatoria import config

## 2. LOAD DATA

Se lee la tabla final de la cadena, `alertas_preguntas.csv` del `04.04`, que reúne la ficha, el
cruce y las alertas y preguntas de cada documento.

In [ ]:
archivo_entrada = (
    config.INTERIM / "4. Preparatoria CCI" / "04.04_alertas_preguntas" / "alertas_preguntas.csv"
)
archivo_salida = config.PROCESSED / "4. Preparatoria CCI.xlsx"

documentos = pd.read_csv(archivo_entrada, encoding="utf-8-sig", dtype=str, keep_default_na=False)
for campo in ["alertas", "preguntas"]:
    documentos[campo] = documentos[campo].map(json.loads)
print(f"{len(documentos)} documentos con la cadena completa (04.01 a 04.04).")

## 3. PROCESS DATA

### Estructura

Los encabezados A–R son los de la hoja `CCI-2026`. Las listas de los desplegables son las del
Input CCI.

In [ ]:
columnas = [
    "Fecha",
    "No.",
    "Operación",
    "Retroalimentación DADMI (insumo Evid+Recom) (1/0)",
    "Tipo de documento",
    "Nombre operación",
    "País",
    "Riesgo",
    "Producto",
    "Producto agrupado",
    "Monto + Financiamiento verde",
    "Financiamiento verde",
    "CT / Antecedentes",
    "Agenda ",
    "Aprendizajes de Evidencia",
    "Recomendaciones 🟢 🟡 🔴 ⚪ ",
    "Alertas",
    "Preguntas y comentarios sugeridos",
    "Revisar",
    "Archivo fuente",
]
listas = {
    "Tipo de documento": [
        "IDO",
        "APROBACIÓN",
        "RENOVACIÓN",
        "REVISIÓN",
        "ENMIENDA",
        "DISPENSA",
        "OTRO",
    ],
    "Riesgo": ["Soberano", "Cuasi soberano", "No soberano"],
    "Producto agrupado": [
        "PPI",
        "PBL",
        "PpR",
        "SWAp",
        "Línea de crédito",
        "Avales y garantías",
        "Préstamo corporativo",
        "Préstamo a largo plazo",
        "Préstamo A/B",
        "Inversión patrimonial",
        "Financiamiento estructurado",
        "Liquidez",
        "Otro",
    ],
    "Financiamiento verde": ["Elegible", "Potencialmente elegible", "No elegible", "NA"],
    "1/0": [1, 0],
}

### Aprendizajes y recomendaciones (O y P)

- **IDO y aprobación**: se copian la evidencia y las recomendaciones del Input CCI; si la
  operación no está allí, las de la última preparatoria en que se vio. Si no hay ninguna, se
  dejan vacías y se pide completarlas.
- **Renovaciones, líneas y demás**: el aporte al desarrollo que leyó el `04.02` (exposición,
  desembolsos y su uso). Si la preparatoria anterior registró aprendizajes, no se copian (suelen
  ser desembolsos de años pasados): se avisa en `Revisar` para agregarlos si siguen vigentes.

In [ ]:
ACEPTADOS = {"cruce", "ajuste manual"}
CON_EVIDENCIA = {"IDO", "APROBACIÓN"}


def fecha_corta(texto):
    return date.fromisoformat(texto).strftime("%d-%m-%Y") if texto else "sin fecha"


def aprendizajes_y_recomendaciones(fila, avisos):
    tiene_input = fila["input_estado"] in ACEPTADOS and fila["input_evidencia"].strip()
    tiene_preparatoria = fila["preparatoria_estado"] in ACEPTADOS
    if fila["tipo_documento"] in CON_EVIDENCIA:
        if tiene_input:
            avisos.append(
                f"Evidencia y Recomendaciones del {fila['input_origen']} "
                f"(IDO del {fecha_corta(fila['input_fecha'])})"
            )
            return fila["input_evidencia"], fila["input_recomendaciones"]
        previas = [
            fila.get("preparatoria_aprendizajes", ""),
            fila.get("preparatoria_recomendaciones", ""),
        ]
        if tiene_preparatoria and any(texto.strip() for texto in previas):
            avisos.append(
                "Evidencia y Recomendaciones de la preparatoria del "
                f"{fecha_corta(fila['preparatoria_fecha'])}"
            )
            return previas[0], previas[1]
        avisos.append(
            "No está en Input CCI: completar Aprendizajes de Evidencia y Recomendaciones si aplica"
        )
        return "", ""
    anterior = fila.get("preparatoria_aprendizajes", "").strip() if tiene_preparatoria else ""
    if anterior:
        avisos.append(
            f"La preparatoria del {fecha_corta(fila['preparatoria_fecha'])} registró aprendizajes "
            "(desembolsos, beneficiarios): agregarlos si siguen vigentes"
        )
    return fila["aporte_desarrollo"].strip(), ""

### Filas del Excel y columna `Revisar`

`Revisar` reúne, separados por punto y coma: los textos que redactó Azure, las dudas que anotó
el `04.02`, la lectura desde imágenes, los cruces "posible" del `04.03`, el origen de O y P, el
semáforo pendiente y la Retroalimentación DADMI por completar.

In [ ]:
def viñetas(lista):
    return "\n".join(f"• {texto}" for texto in lista)


def armar_fila(fila):
    avisos = ["Textos de C, K–N, Q y R redactados con LLM: validar"]
    if fila["dudas"].strip():
        avisos.append(f"Dudas de la lectura: {fila['dudas'].strip()}")
    if fila["escaneado"] == "True":
        avisos.append("PDF escaneado, leído desde imágenes: verificar cifras")
    if not fila["numero_agenda"]:
        avisos.append("Completar No. (el archivo no empieza con número)")
    for fuente, etiqueta in [("input", "Input CCI"), ("preparatoria", "preparatoria anterior")]:
        if fila[f"{fuente}_estado"] == "posible":
            avisos.append(
                f"Posible cruce con {etiqueta} no usado: {fila[f'{fuente}_nombre'][:80]} "
                "(confirmar en ajustes_cruce del 04.03)"
            )

    aprendizajes, recomendaciones = aprendizajes_y_recomendaciones(fila, avisos)
    if recomendaciones.strip():
        avisos.append("Semáforo de recomendaciones pendiente")

    preguntas = viñetas(fila["preguntas"])
    if fila["priorizacion"].strip():
        preguntas = f"{fila['priorizacion'].strip()}\n\nPreguntas sugeridas:\n{preguntas}"

    if fila["insumo_estado"] in ACEPTADOS:
        avisos.append(
            f"Hay insumo Evid+Recom ({fila['insumo_archivo_insumo']}): "
            "Retroalimentación DADMI sugerida 1"
        )
    avisos.append("Completar Retroalimentación DADMI")

    return {
        "Fecha": date.fromisoformat(fila["fecha_sesion"]) if fila["fecha_sesion"] else None,
        "No.": int(fila["numero_agenda"]) if fila["numero_agenda"] else None,
        "Operación": (
            f"{fila['tipo_documento']}: {fila['nombre_operacion']}\n\n{fila['operacion']}".strip()
        ),
        "Retroalimentación DADMI (insumo Evid+Recom) (1/0)": None,
        "Tipo de documento": fila["tipo_documento"],
        "Nombre operación": fila["nombre_operacion"],
        "País": fila["pais"],
        "Riesgo": fila["riesgo"],
        "Producto": fila["producto"],
        "Producto agrupado": fila["producto_agrupado"],
        "Monto + Financiamiento verde": fila["monto"],
        "Financiamiento verde": fila["financiamiento_verde"],
        "CT / Antecedentes": fila["ct_antecedentes"],
        "Agenda ": fila["agenda"],
        "Aprendizajes de Evidencia": aprendizajes,
        "Recomendaciones 🟢 🟡 🔴 ⚪ ": recomendaciones,
        "Alertas": viñetas(fila["alertas"]),
        "Preguntas y comentarios sugeridos": preguntas,
        "Revisar": "; ".join(avisos),
        "Archivo fuente": f"{fila['sesion']}/{fila['archivo']}",
    }

### Revisión antes de guardar

Las filas se ordenan por fecha de la sesión y número en la agenda. Los textos largos se
abrevian solo en esta vista.

In [ ]:
tabla = pd.DataFrame(
    [armar_fila(fila) for fila in documentos.to_dict("records")], columns=columnas
)
tabla = tabla.sort_values(["Fecha", "No."], kind="stable").reset_index(drop=True)
tabla.index = pd.RangeIndex(1, len(tabla) + 1)
with pd.option_context(
    "display.max_columns", None, "display.max_rows", None, "display.max_colwidth", 60
):
    display(tabla)

## 4. SAVE DATA

Se crean las hojas `Preparatoria CCI` y `Listas`, los desplegables y el formato, como en el
`02.05`. Se inmoviliza `D2` para mantener visibles la fila 1 y las columnas A–C. Solo se
reemplaza `data/processed/4. Preparatoria CCI.xlsx`.

In [ ]:
libro = Workbook()
hoja = libro.active
hoja.title = "Preparatoria CCI"
hoja_listas = libro.create_sheet("Listas")
hoja.append(columnas)
for valores in tabla.itertuples(index=False, name=None):
    hoja.append([None if pd.isna(valor) or valor == "" else valor for valor in valores])

rangos_listas = {}
for numero, (nombre, opciones) in enumerate(listas.items(), start=1):
    hoja_listas.cell(1, numero, nombre)
    for fila, opcion in enumerate(opciones, start=2):
        hoja_listas.cell(fila, numero, opcion)
    letra = get_column_letter(numero)
    nombre_rango = f"lista_{numero}"
    referencia = f"'Listas'!${letra}$2:${letra}${len(opciones) + 1}"
    libro.defined_names.add(DefinedName(nombre_rango, attr_text=referencia))
    rangos_listas[nombre] = nombre_rango

desplegables = {
    "Retroalimentación DADMI (insumo Evid+Recom) (1/0)": "1/0",
    "Tipo de documento": "Tipo de documento",
    "Riesgo": "Riesgo",
    "Producto agrupado": "Producto agrupado",
    "Financiamiento verde": "Financiamiento verde",
}
for columna, nombre_lista in desplegables.items():
    validacion = DataValidation(
        type="list",
        formula1=f"={rangos_listas[nombre_lista]}",
        allow_blank=True,
        showErrorMessage=True,
        errorStyle="stop",
        errorTitle="Valor fuera de la lista",
        error="Selecciona un valor de la lista desplegable.",
    )
    letra = get_column_letter(columnas.index(columna) + 1)
    validacion.add(f"{letra}2:{letra}{max(2, hoja.max_row)}")
    hoja.add_data_validation(validacion)

In [ ]:
relleno_encabezado = PatternFill("solid", fgColor="1F3864")
fuente_encabezado = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
for pagina in (hoja, hoja_listas):
    for celda in pagina[1]:
        celda.fill = relleno_encabezado
        celda.font = fuente_encabezado
        celda.alignment = Alignment(wrap_text=True, vertical="center")
    for fila in pagina.iter_rows(min_row=2):
        for celda in fila:
            celda.alignment = Alignment(wrap_text=True, vertical="top")

anchos = {
    "Fecha": 12,
    "No.": 5,
    "Operación": 70,
    "Retroalimentación DADMI (insumo Evid+Recom) (1/0)": 14,
    "Nombre operación": 40,
    "Producto": 25,
    "Monto + Financiamiento verde": 32,
    "CT / Antecedentes": 30,
    "Agenda ": 32,
    "Aprendizajes de Evidencia": 60,
    "Recomendaciones 🟢 🟡 🔴 ⚪ ": 60,
    "Alertas": 55,
    "Preguntas y comentarios sugeridos": 55,
    "Revisar": 45,
    "Archivo fuente": 40,
}
for numero, nombre in enumerate(columnas, start=1):
    hoja.column_dimensions[get_column_letter(numero)].width = anchos.get(nombre, 16)
for numero in range(1, len(listas) + 1):
    hoja_listas.column_dimensions[get_column_letter(numero)].width = 28

hoja.row_dimensions[1].height = 45
for numero in range(2, hoja.max_row + 1):
    hoja.row_dimensions[numero].height = 150
    hoja.cell(numero, 1).number_format = "yyyy-mm-dd"
hoja.auto_filter.ref = f"A1:{get_column_letter(len(columnas))}{hoja.max_row}"
hoja.freeze_panes = "D2"
hoja_listas.freeze_panes = "A2"

In [ ]:
archivo_salida.parent.mkdir(parents=True, exist_ok=True)
libro.save(archivo_salida)
libro.close()
print(f"Guardado: {archivo_salida.relative_to(ROOT)} ({len(tabla)} documentos)")